In [ ]:
from functools import lru_cache

import numpy as np
import pandas as pd
from rdkit import Chem
from descriptastorus.descriptors import rdNormalizedDescriptors
import tqdm.auto as tqdm

In [ ]:
df = pd.read_csv(
    "Cleaned_VLE_Data_with_Smiles_and_Mols.csv"
)

In [ ]:
# To ensure mol are mol objects
df["mol1"] = df["Smiles 1"].apply(Chem.MolFromSmiles)
df["mol2"] = df["Smiles 2"].apply(Chem.MolFromSmiles)

In [ ]:
df.head()

In [ ]:
generator = rdNormalizedDescriptors.RDKit2DNormalized()
print(generator.columns)

In [ ]:
# 1. Define a wrapper function and apply the cache to IT


@lru_cache(maxsize=None)
def get_cached_descriptors(mol_input):
    # This function simply calls your generator
    # We hardcode 'None' here since you used it in your original snippet
    return generator.calculateMol(mol_input, None)


# 2. Now run your loops calling the CACHED function
# The first time a specific molecule is seen, it runs the generator.
# The second time (e.g., if the same molecule is in mol2), it returns
# instantly.


print("Processing Mol 1...")
x_1 = np.stack([get_cached_descriptors(m)
    for m in tqdm.tqdm(df["mol1"].tolist())])

print("Processing Mol 2...")
x_2 = np.stack([get_cached_descriptors(m)
    for m in tqdm.tqdm(df["mol2"].tolist())])

# Optional: Check cache stats to see how much time you saved
print(f"\nCache Info: {get_cached_descriptors.cache_info()}")

print(x_1.shape)
print(x_2.shape)

In [ ]:
np.save("descriptors_for_comp_1.npy", x_1)
np.save("descriptors_for_comp_2.npy", x_2)

In [ ]:
df_mol1 = pd.DataFrame(x_1, columns=generator.columns)
df_mol2 = pd.DataFrame(x_2, columns=generator.columns)
df = pd.concat([df, df_mol1, df_mol2], axis=1)
print(df.shape)
df.head()

In [ ]:
df.dropna(inplace=True)
df.shape
df.to_csv(
    "Cleaned_VLE_Data_with_Smiles_and_Mols_and_Descriptors.csv",
    index=False
)

In [ ]:
print(df.shape)